# CMPT 211: Numerical Libraries - NumPy and SciPy
Today we transition from standard Python lists to high-performance numerical arrays.
* **NumPy:** The core library for $N$-dimensional arrays and element-wise math.
* **SciPy:** Built directly on top of NumPy, providing specialized, heavily optimized algorithms (often using underlying C/Fortran libraries like BLAS and LAPACK).

In [ ]:
import numpy as np
import scipy
import scipy.linalg as slinalg
import scipy.stats as stats
import scipy.signal as signal
import matplotlib.pyplot as plt

print(f"NumPy Version: {np.__version__}")
print(f"SciPy Version: {scipy.__version__}")

### 1.1 Array Declarations & Signatures
NumPy arrays (`ndarray`) are the universal data structure for both libraries. They require contiguous memory and a single data type, making them exponentially faster than Python lists.

In [ ]:
# 1D Vector and 2D Matrix declarations
vector_x = np.array([1.0, 2.0, 3.0])
matrix_A = np.array([[1, 2], [3, 4]], dtype=np.float64)

# Generating structural arrays instantly
zeros_matrix = np.zeros((3, 3))
identity_matrix = np.eye(3)

print("Vector:\n", vector_x)
print("Identity Matrix:\n", identity_matrix)

In [ ]:
# Array Introspection: Checking the physical memory structure
print("Shape (Rows, Columns):", matrix_A.shape)
print("Total Elements:", matrix_A.size)
print("Data Type:", matrix_A.dtype)

### 2.1 NumPy vs. SciPy: Random Numbers (Normal Distribution)
Both libraries can generate random numbers.
* **NumPy** is designed for fast, bulk array generation.
* **SciPy** treats distributions as formal mathematical objects, allowing you to generate numbers *and* calculate exact probabilities (PDF/CDF).

In [ ]:
# APPLES-TO-APPLES: Generating 5 random samples from a Standard Normal Distribution (Mean=0, Std=1)

# NumPy Approach: Fast bulk generation
np_random_array = np.random.normal(loc=0.0, scale=1.0, size=5)
print("NumPy Randoms:", np_random_array)

# SciPy Approach: Using the formal statistical object
scipy_norm_dist = stats.norm(loc=0.0, scale=1.0)
scipy_random_array = scipy_norm_dist.rvs(size=5)
print("SciPy Randoms:", scipy_random_array)

### 2.2 NumPy vs. SciPy: Linear Algebra
Both have a `linalg` module. NumPy's is good for basic operations. SciPy's `linalg` is strictly preferred in engineering because it is compiled against highly optimized Fortran libraries (LAPACK/BLAS), making it faster and more numerically stable for complex matrices.

In [ ]:
# APPLES-TO-APPLES: Matrix Inversion
square_matrix = np.array([[4, 7], [2, 6]])

# NumPy Inversion
np_inv = np.linalg.inv(square_matrix)

# SciPy Inversion (Standard practice for heavy computations)
sp_inv = slinalg.inv(square_matrix)

print("NumPy Inverse:\n", np_inv)
print("\nSciPy Inverse:\n", sp_inv)

In [ ]:
# Helper function for perceptual color-coded matrices
def plot_matrix(matrix, title):
    plt.figure(figsize=(6, 5))
    # 'turbo' provides a highly distinct perceptual rainbow palette
    plt.imshow(matrix, cmap='turbo', interpolation='nearest')
    plt.colorbar(label='Magnitude')
    plt.title(title)

    # Overlay the numerical values line-by-line
    for i in range(matrix.shape[0]):
        for j in range(matrix.shape[1]):
            val = matrix[i, j]
            # Switch text color to white for dark background values
            text_color = "white" if val < matrix.max() * 0.4 else "black"
            plt.text(j, i, f"{val:.2f}", ha="center", va="center", color=text_color)
    plt.show()

### 3.1 Transparent Positive Definite Matrices
A strictly Symmetric Positive Definite (SPD) matrix has positive eigenvalues. We will construct a classic 1D discrete Laplacian matrix (2 on the diagonal, -1 on the sub/super diagonals), which is mathematically guaranteed to be SPD.

In [ ]:
# Constructing a 4x4 Symmetric Positive Definite Matrix
A = np.array([
    [ 2, -1,  0,  0],
    [-1,  2, -1,  0],
    [ 0, -1,  2, -1],
    [ 0,  0, -1,  2]
], dtype=float)

plot_matrix(A, "Symmetric Positive Definite Matrix (A)")

# Proof: All eigenvalues must be > 0
eigenvalues = slinalg.eigvals(A)
print("Eigenvalues (Proof of Positive Definiteness):", np.real(eigenvalues))

In [ ]:
# Transpose (Symmetric matrices equal their transpose: A == A.T)
print("Original:\n", A)
print("\nTransposed:\n", A.T)

# Matrix Norm (Magnitude of the matrix)
print(f"\nFrobenius Norm of A: {slinalg.norm(A):.2f}")

In [ ]:
# Matrix-Vector and Matrix-Matrix Multiplication
x = np.array([1, 2, 3, 4])
B = np.eye(4) * 0.5

# Dot product / Matrix-Vector mult
Ax = A @ x
print("A @ x =", Ax)

# Matrix-Matrix mult
AB = A @ B
plot_matrix(AB, "Matrix Multiplication (A @ B)")

### 3.2 Classic Signal Processing: Convolution Modes
Convolution slides a kernel over a signal. The `mode` dictates how boundaries are handled.

In [ ]:
signal_1d = np.array([1, 2, 3, 4, 5])
kernel = np.array([0.5, 1.0, 0.5])

# Full: Pads edges with zeros, output is larger than input
print("Full: ", signal.convolve(signal_1d, kernel, mode='full'))

# Same: Output matches input size
print("Same: ", signal.convolve(signal_1d, kernel, mode='same'))

# Valid: Only computes where kernel fully overlaps the signal
print("Valid:", signal.convolve(signal_1d, kernel, mode='valid'))

### 4.1 1D Calculus (Finite Differences & Integrals)

In [ ]:
# 1D Array representing y = x^2
x_vals = np.linspace(0, 5, 6)
y_vals = x_vals ** 2

# Discrete Derivative (Forward Difference)
dy = np.diff(y_vals)
dx = np.diff(x_vals)
print("1D Derivative (dy/dx):", dy / dx)

# Trapezoidal Integral
from scipy.integrate import trapezoid
print(f"Trapezoidal Integral of x^2 from 0 to 5: {trapezoid(y_vals, x_vals):.2f}")

### 4.2 2D Image Gradients (Color-Coded)
Let's generate a 2D synthetic image (a Gaussian pulse) and compute its gradient. This uses classic signal processing finite differences, visualizing the directional change in pixel intensity.

In [ ]:
# Generate a classic 2D Gaussian Image (7x7)
x, y = np.meshgrid(np.linspace(-1, 1, 7), np.linspace(-1, 1, 7))
gaussian_image = np.exp(-(x**2 + y**2) * 2)

plot_matrix(gaussian_image, "2D Image: Gaussian Pulse")

In [ ]:
# Compute the 2D Gradient using central finite differences
# Returns a list of two arrays: [Gradient in Y (rows), Gradient in X (cols)]
grad_y, grad_x = np.gradient(gaussian_image)

plot_matrix(grad_x, "Image Gradient (X-Direction / Columns)")

### 4.3 The Mathematical Unification: Circulant Matrices
In signal processing, performing a convolution with a finite difference filter `[-1, 1, 0]` is mathematically identical to performing a matrix-vector multiplication where the matrix is **Circulant**.

Notice the beautiful banded, shifted diagonal structure!

In [ ]:
# Creating a Circulant Matrix for a 1D finite difference kernel
# Kernel: [-1, 1, 0, 0, 0]
column_vector = np.array([-1, 0, 0, 0, 1])

# Generate the circulant matrix
circulant_matrix = slinalg.circulant(column_vector)

plot_matrix(circulant_matrix, "Circulant Matrix (Finite Difference Operator)")